# 第 58 章：Capstone 社群记忆与长期维护账本

这个 notebook 用一个极小的 memory-ledger table，对比“只按 continuity priority 排序”的 naive baseline 和“先检查边界、owner、陈旧度与刷新节奏”的长期维护 workflow。

In [ ]:
from __future__ import annotations

import csv
import platform
from collections import Counter
from pathlib import Path

DATA_PATH = Path('../../data/small/capstone_community_memory_maintenance_ledger_demo.csv')


def load_items(path):
    rows = []
    with path.open(encoding='utf-8', newline='') as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            row['continuity_priority_score'] = float(row['continuity_priority_score'])
            rows.append(row)
    return rows


def ordered_counts(rows, field):
    counts = Counter(row[field] for row in rows)
    return {key: counts[key] for key in sorted(counts)}


rows = load_items(DATA_PATH)
print(f'Loaded {len(rows)} ledger items from {DATA_PATH.name}')
print('Python version:', platform.python_version())
print('Retention value:', ordered_counts(rows, 'retention_value_status'))
print('Update frequency:', ordered_counts(rows, 'update_frequency_status'))
print('Owner status:', ordered_counts(rows, 'owner_status'))
print('Access boundary:', ordered_counts(rows, 'access_boundary_status'))
print('Staleness:', ordered_counts(rows, 'staleness_status'))
print('Reference route:', ordered_counts(rows, 'reference_route'))


In [ ]:
reference_ready_ids = sorted(
    row['ledger_id']
    for row in rows
    if row['reference_route'] == 'ledger_ready'
)
top4_priority = sorted(
    rows,
    key=lambda row: row['continuity_priority_score'],
    reverse=True,
)[:4]
baseline_ready_hits = sum(
    row['reference_route'] == 'ledger_ready'
    for row in top4_priority
)
baseline_not_ready_hits = sum(
    row['reference_route'] != 'ledger_ready'
    for row in top4_priority
)
baseline_ready_precision = baseline_ready_hits / len(top4_priority)
baseline_not_ready_intrusion = baseline_not_ready_hits / len(top4_priority)

print('Reference ledger-ready items:', reference_ready_ids)
print('Top-4 by continuity priority:')
for row in top4_priority:
    print(
        f"  {row['ledger_id']}: priority={row['continuity_priority_score']:.2f}, "
        f"route={row['reference_route']}, area={row['memory_area']}"
    )
print('Baseline ledger precision:', round(baseline_ready_precision, 3))
print('Baseline not-ready intrusion:', round(baseline_not_ready_intrusion, 3))


In [ ]:
def route_ledger_item(row):
    if row['access_boundary_status'] not in {'internal', 'public'}:
        return 'review_access_boundary', 'access boundary still needs review before long-term retention'
    if row['owner_status'] != 'assigned':
        return 'assign_ledger_owner', 'ledger item has no maintenance owner yet'
    if row['staleness_status'] != 'current':
        return 'refresh_stale_memory', 'ledger memory is stale and should be refreshed'
    if row['update_frequency_status'] != 'scheduled':
        return 'schedule_ledger_refresh', 'refresh cadence is not scheduled yet'
    return 'ledger_ready', 'item has boundary, owner, current content, and a refresh cadence'


routed_rows = []
for row in rows:
    route, reason = route_ledger_item(row)
    routed = dict(row)
    routed['route'] = route
    routed['reason'] = reason
    routed_rows.append(routed)

print('Memory-ledger workflow routes:')
for row in routed_rows:
    print(
        f"  {row['ledger_id']}: route={row['route']}, "
        f"reference={row['reference_route']}, reason={row['reason']}"
    )


In [ ]:
ready_queue = [row for row in routed_rows if row['route'] == 'ledger_ready']
schedule_queue = [row for row in routed_rows if row['route'] == 'schedule_ledger_refresh']
owner_queue = [row for row in routed_rows if row['route'] == 'assign_ledger_owner']
stale_queue = [row for row in routed_rows if row['route'] == 'refresh_stale_memory']
boundary_queue = [row for row in routed_rows if row['route'] == 'review_access_boundary']

workflow_accuracy = sum(
    row['route'] == row['reference_route']
    for row in routed_rows
) / len(routed_rows)
ready_precision = sum(
    row['reference_route'] == 'ledger_ready'
    for row in ready_queue
) / max(len(ready_queue), 1)

print('Ledger-ready queue:', [row['ledger_id'] for row in ready_queue])
print('Schedule-refresh queue:', [row['ledger_id'] for row in schedule_queue])
print('Assign-owner queue:', [row['ledger_id'] for row in owner_queue])
print('Refresh-stale queue:', [row['ledger_id'] for row in stale_queue])
print('Review-boundary queue:', [row['ledger_id'] for row in boundary_queue])
print('Workflow route accuracy:', round(workflow_accuracy, 3))
print('Structured ledger precision:', round(ready_precision, 3))


In [ ]:
def ledger_steps(row):
    steps = []
    if row['access_boundary_status'] not in {'internal', 'public'}:
        steps.append('先复核访问边界，决定该条目应内部保留、公开还是受限访问。')
    if row['owner_status'] != 'assigned':
        steps.append('指定 ledger owner，并明确谁负责刷新与淘汰。')
    if row['staleness_status'] != 'current':
        steps.append('刷新陈旧记忆：补上下学期仍然可信的版本和上下文。')
    if row['update_frequency_status'] != 'scheduled':
        steps.append('安排刷新节奏：每学期、每学年或按事件触发更新。')
    return steps or ['可以进入长期维护账本；保留 owner、更新时间和引用入口。']


for row in routed_rows:
    if row['route'] != 'ledger_ready':
        print(f"\n{row['ledger_id']} -> {row['route']} ({row['memory_area']})")
        for step in ledger_steps(row):
            print(' -', step)


In [ ]:
ledger_outline = [
    'Memory item: what experience or lesson this entry preserves',
    'Retention reason: why it matters to future cohorts or staff',
    'Owner: who refreshes, reviews, or retires the item',
    'Refresh cadence: per semester, per year, or event-triggered',
    'Boundary: internal, public, restricted, or role-limited',
    'Reference path: where this memory should be cited in handoff, reboot, or release material',
]

ledger_assistant_prompt = '''你是我的 capstone 长期维护账本助手。

任务：
1. 先阅读 memory-ledger table，不要只按 continuity priority 排序；
2. 先检查 access boundary；
3. 再检查 owner、staleness 和 update cadence；
4. 把每个条目 route 到 ledger_ready、schedule_ledger_refresh、
   assign_ledger_owner、refresh_stale_memory 或 review_access_boundary；
5. 对每个非 ready 条目输出长期维护前 checklist。

输出格式：
- Ledger-ready items
- Schedule-refresh items
- Assign-owner items
- Refresh-stale items
- Review-boundary items
- Ledger checklist by item
'''

print('Community memory ledger outline:')
for item in ledger_outline:
    print(' -', item)

print('\nAssistant prompt:')
print(ledger_assistant_prompt)


In [ ]:
ledger_snapshot = {
    'dataset': DATA_PATH.name,
    'n_ledger_items': len(rows),
    'baseline_top4_ledger_precision': round(baseline_ready_precision, 3),
    'baseline_not_ready_intrusion': round(baseline_not_ready_intrusion, 3),
    'workflow_route_accuracy': round(workflow_accuracy, 3),
    'ledger_ready': [row['ledger_id'] for row in ready_queue],
    'schedule_ledger_refresh': [row['ledger_id'] for row in schedule_queue],
    'assign_ledger_owner': [row['ledger_id'] for row in owner_queue],
    'refresh_stale_memory': [row['ledger_id'] for row in stale_queue],
    'review_access_boundary': [row['ledger_id'] for row in boundary_queue],
    'python_version': platform.python_version(),
}

print('Community memory ledger snapshot:')
for key, value in ledger_snapshot.items():
    print(f'  {key}: {value}')
